# Fase 1 — Comprensión del problema y línea base
**Proyecto Integrador · Deep Learning · Universidad Externado de Colombia**
**Autor:** Nelson Andrey García Patarroyo

**Objetivo de este cuaderno.** Construir el modelo de referencia (línea base) contra el que se mide la arquitectura profunda en las Fases 2 y 3: **TF-IDF de unigramas y bigramas + regresión logística**.

**Datos.** `fake_news_dataset.csv`: 134 afirmaciones que circularon en Colombia y fueron verificadas por *fact-checkers* (Colombiacheck y otros).
**Convención de etiquetas:** en la columna `fake_news`, `True` significa que la noticia es **verdadera** (verificado contra los enlaces de `link_verificador`). Por eso se define `y = 1` verdadera y `y = 0` falsa.

**Métrica principal:** F1 de la clase «verdadera»; se reportan además exactitud, precisión, exhaustividad y MCC (el MCC corrige el desbalance 74/60).

In [1]:
# 1. Librerías y configuración (semilla fija para que todo sea reproducible)
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             matthews_corrcoef, confusion_matrix, make_scorer)

SEED = 42                       # misma semilla en las tres fases → misma partición 107/27
DATA = "fake_news_dataset.csv"  # el archivo debe estar junto al cuaderno (en Colab: súbelo al panel de archivos)

In [2]:
# 2. Carga del conjunto de datos (separador ';' y codificación utf-8 con BOM)
df = pd.read_csv(DATA, sep=";", encoding="utf-8-sig")
df = df.loc[:, ~df.columns.str.match("Unnamed")]          # columnas vacías al final del CSV
df["y"] = df["fake_news"].astype(str).str.strip().str.lower().eq("true").astype(int)   # 1 = verdadera, 0 = falsa
print(df.shape)
print(df["y"].value_counts().rename({0: "falsa (0)", 1: "verdadera (1)"}))

(134, 14)
y
falsa (0)        74
verdadera (1)    60
Name: count, dtype: int64


## Exploración mínima y riesgos de fuga de información
Solo el **texto** (título + cuerpo) entra al modelo. Las columnas de metadatos se revisan para detectar atajos:
- `tipo_noticia` y `tipologia_8P` solo están diligenciadas para las noticias **falsas** → revelan la etiqueta y **nunca** se usan como entrada.
- `fuente` y `medio` no entran al modelo, pero están muy asociadas a la clase; en la Fase 3 se usan para auditar si el modelo aprende el origen del texto en lugar de su contenido.

In [3]:
# 3. ¿Qué metadatos están asociados a la etiqueta? (porcentaje de verdaderas por grupo)
for col in ["fuente", "medio"]:
    t = pd.crosstab(df[col].astype(str).str.strip(), df["y"])
    t["% verdaderas"] = (100 * t[1] / t.sum(axis=1)).round(0)
    print(t, "\n")
print("tipo_noticia diligenciado por clase:\n", df.groupby("y")["tipo_noticia"].apply(lambda s: s.notna().sum()))

y                       0   1  % verdaderas
fuente                                     
Anónima                19   2          10.0
Identificada           38  24          39.0
Medio de comunicación  10  29          74.0
Oficial                 0   5         100.0
Suplantada              7   0           0.0 

y            0   1  % verdaderas
medio                           
Facebook    25   8          24.0
Instagram    3   5          62.0
Página web   8  20          71.0
Telegram     1   0           0.0
Twitter     37  27          42.0 

tipo_noticia diligenciado por clase:
 y
0    74
1     0
Name: tipo_noticia, dtype: int64


In [4]:
# 4. Preparación del texto para TF-IDF: minúsculas, sin URL y sin signos (se conservan tildes y ñ)
def clean_text(t: str) -> str:
    t = str(t).lower()
    t = re.sub(r"http\S+|www\.\S+", " ", t)
    t = re.sub(r"[^a-záéíóúñü0-9\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

df["text"] = (df["titulo"].fillna("") + " " + df["texto"].fillna("")).apply(clean_text)
df["palabras"] = df["text"].str.split().str.len()
print(df.groupby("y")["palabras"].describe().round(0))

   count  mean   std   min   25%   50%    75%    max
y                                                   
0   74.0  86.0  87.0  11.0  35.0  46.0  105.0  432.0
1   60.0  83.0  69.0  12.0  38.0  48.0  104.0  395.0


In [5]:
# 5. Partición estratificada 80/20 (107 desarrollo / 27 prueba). La prueba NO se toca hasta el final.
Xtr, Xte, ytr, yte = train_test_split(df["text"], df["y"], test_size=0.2, stratify=df["y"], random_state=SEED)
print(f"Desarrollo: {len(Xtr)} | Prueba: {len(Xte)} | % verdaderas en prueba: {yte.mean():.1%}")

Desarrollo: 107 | Prueba: 27 | % verdaderas en prueba: 44.4%


In [6]:
# 6. Línea base: TF-IDF (1-2 gramas, min_df=2) + regresión logística con pesos balanceados.
#    Al ir dentro de un Pipeline, el vocabulario TF-IDF se ajusta SOLO con cada pliegue de entrenamiento (sin fuga).
def make_baseline():
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)),
    ])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = {"accuracy": "accuracy", "precision": "precision", "recall": "recall", "f1": "f1",
           "mcc": make_scorer(matthews_corrcoef)}
cv = cross_validate(make_baseline(), Xtr, ytr, cv=skf, scoring=scoring)
cv_table = pd.DataFrame({m: [cv[f"test_{m}"].mean(), cv[f"test_{m}"].std()] for m in scoring}, index=["media", "desv."]).round(3)
cv_table

,accuracy,precision,recall,f1,mcc
media,0.663,0.628,0.664,0.639,0.327
desv.,0.059,0.078,0.084,0.043,0.106


In [7]:
# 7. Evaluación única sobre la prueba (27 casos)
pipe = make_baseline().fit(Xtr, ytr)
pred = pipe.predict(Xte)
res = dict(accuracy=accuracy_score(yte, pred), precision=precision_score(yte, pred), recall=recall_score(yte, pred),
           f1=f1_score(yte, pred), mcc=matthews_corrcoef(yte, pred))
print({k: round(v, 3) for k, v in res.items()})
print("Matriz de confusión [filas = real, columnas = predicción; orden falsa, verdadera]:\n", confusion_matrix(yte, pred))
print("Referencia trivial «todo verdadero»: F1 =", round(f1_score(yte, np.ones(len(yte))), 3), "| MCC = 0")

{'accuracy': 0.556, 'precision': 0.5, 'recall': 0.417, 'f1': 0.455, 'mcc': 0.086}
Matriz de confusión [filas = real, columnas = predicción; orden falsa, verdadera]:
 [[10  5]
 [ 7  5]]
Referencia trivial «todo verdadero»: F1 = 0.615 | MCC = 0


In [8]:
# 8. ¿Qué n-gramas usa la línea base? (coeficientes de la regresión logística)
vocab = np.array(pipe.named_steps["tfidf"].get_feature_names_out())
coef = pipe.named_steps["clf"].coef_[0]
order = np.argsort(coef)
print("Hacia «falsa»:     ", list(vocab[order[:15]]))
print("Hacia «verdadera»: ", list(vocab[order[-15:][::-1]]))

Hacia «falsa»:      ['que', 'lo', 'petro', 'cambio', 'última hora', 'hora', 'hay', 'última', 'np el', 'es', 'les', 'por', 'más', 'guerrilleros', 'información']
Hacia «verdadera»:  ['de', 'el', 'en', 'una', 'en el', 'de la', 'millones de', 'fracking', 'votó', '2023', 'producto', 'misma', 'había', 'de los', 'la paz']


## Lectura
- La línea base obtiene en validación cruzada **F1 ≈ 0,64** y en la prueba **F1 = 0,455 (MCC = 0,086)**: la prueba de 27 casos es muy pequeña y ruidosa.
- El predictor trivial «todo verdadero» saca F1 = 0,615 en la prueba: por eso el F1 se reporta siempre junto al MCC.
- Esta es la referencia con la que se comparan RoBERTa-BNE (Fase 2) y el modelo final auditado (Fase 3).